# 1. Initial Setup and Data Loading

In [58]:
from google.colab import drive
import os

drive.mount('/content/drive')

VOX_PATH = "/content/drive/MyDrive/Capstone Project/timit/timit"

audio_files = []

for root, dirs, files in os.walk(VOX_PATH):
    for file in files:
        if file.endswith(".wav"):
            audio_files.append(os.path.join(root, file))

print("Total original audio files found:", len(audio_files))


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Total original audio files found: 160


# 2. Install Required Libraries

In [ ]:
!pip install pydub audiomentations soundfile librosa scikit-learn
print("Installed pydub, audiomentations, soundfile, librosa, and scikit-learn libraries.")

Installed pydub, audiomentations, soundfile, librosa, and scikit-learn libraries.


# 3. Audio Augmentation

In [ ]:
import numpy as np
from audiomentations import Compose, AddGaussianNoise, PitchShift, TimeStretch, Normalize
from pydub import AudioSegment

def create_augmentation_pipeline():
    """Creates an audio augmentation pipeline using audiomentations."""
    pipeline = Compose(
        transforms=[
            AddGaussianNoise(min_amplitude=0.001, max_amplitude=0.015, p=0.5),
            PitchShift(min_semitones=-4, max_semitones=4, p=0.5),
            TimeStretch(min_rate=0.8, max_rate=1.2, p=0.5),
            Normalize(p=0.5),
        ]
    )
    return pipeline

def augment_and_save_audio(audio_path, output_dir, pipeline, n_augmentations):
    """Augments an audio file and saves multiple variations.

    Args:
        audio_path (str): Path to the original audio file.
        output_dir (str): Base directory to save augmented files.
        pipeline (audiomentations.Compose): The augmentation pipeline.
        n_augmentations (int): Number of augmented variations to generate.
    """
    original_filename = os.path.basename(audio_path)
    base_name, ext = os.path.splitext(original_filename)

    try:
        audio_segment = AudioSegment.from_wav(audio_path)
    except Exception as e:
        print(f"Error loading {audio_path}: {e}. Skipping augmentation for this file.")
        return

    samples = np.array(audio_segment.get_array_of_samples(), dtype=np.float32)
    sample_rate = audio_segment.frame_rate

    specific_output_dir = os.path.join(output_dir, os.path.relpath(os.path.dirname(audio_path), VOX_PATH), base_name)
    os.makedirs(specific_output_dir, exist_ok=True)

    for i in range(n_augmentations):
        try:
            augmented_samples = pipeline(samples=samples, sample_rate=sample_rate)

            if augmented_samples.dtype == np.int16:
                sample_width = 2
            elif augmented_samples.dtype == np.int32 or augmented_samples.dtype == np.float32:
                sample_width = 4
            else:
                sample_width = audio_segment.sample_width

            augmented_audio = AudioSegment(augmented_samples.tobytes(),
                                         frame_rate=sample_rate,
                                         sample_width=sample_width,
                                         channels=audio_segment.channels)

            output_filename = f"{base_name}_aug_{i+1}{ext}"
            output_filepath = os.path.join(specific_output_dir, output_filename)

            augmented_audio.export(output_filepath, format="wav")

        except Exception as e:
            print(f"Error augmenting and saving {audio_path} (augmentation {i+1}): {e}")
            continue

    print(f"Generated {n_augmentations} augmented versions for {original_filename} in {specific_output_dir}")

In [ ]:
OUTPUT_DIR = "/content/drive/MyDrive/Capstone Project/timit_augmented"
os.makedirs(OUTPUT_DIR, exist_ok=True)

pipeline = create_augmentation_pipeline()
n_augmentations_per_file = 40

print(f"Starting augmentation process for {len(audio_files)} original files.")
for i, audio_file in enumerate(audio_files):
    print(f"Processing file {i+1}/{len(audio_files)}: {audio_file}")
    augment_and_save_audio(audio_file, OUTPUT_DIR, pipeline, n_augmentations_per_file)

print(f"Augmentation process completed for all {len(audio_files)} original files.")

Starting augmentation process for 160 original files.
Processing file 1/160: /content/drive/MyDrive/Capstone Project/timit/timit/dr7-madd0/sa2.wav


KeyboardInterrupt: 

In [ ]:
total_augmented_files = 0
expected_files = len(audio_files) * n_augmentations_per_file # 160 original files * 40 augmentations each

for root, dirs, files in os.walk(OUTPUT_DIR):
    for file in files:
        if file.endswith(".wav"):
            total_augmented_files += 1

print(f"Total augmented audio files found: {total_augmented_files}")

if total_augmented_files == expected_files:
    print(f"Verification successful: {total_augmented_files} files match the expected {expected_files} files.")
else:
    print(f"Verification failed: Found {total_augmented_files} files, but expected {expected_files} files.")


Total augmented audio files found: 6400
Verification successful: 6400 files match the expected 6400 files.


# 4. Audio Splicing

In [ ]:
AUGMENTED_VOX_PATH = "/content/drive/MyDrive/Capstone Project/timit_augmented"

augmented_audio_files = []

for root, dirs, files in os.walk(AUGMENTED_VOX_PATH):
    for file in files:
        if file.endswith(".wav"):
            augmented_audio_files.append(os.path.join(root, file))

print(f"Total augmented audio files found for splicing: {len(augmented_audio_files)}")

def splice_and_save_audio(audio_file_1_path, audio_file_2_path, output_dir, unique_id):
    """Loads two audio files, concatenates them, and saves the spliced audio.

    Args:
        audio_file_1_path (str): Path to the first audio file.
        audio_file_2_path (str): Path to the second audio file.
        output_dir (str): Base directory where the spliced audio will be saved.
        unique_id (str): A unique identifier for the output file and its subdirectory.
    """
    try:
        audio1 = AudioSegment.from_wav(audio_file_1_path)
        audio2 = AudioSegment.from_wav(audio_file_2_path)

        if audio1.frame_rate != audio2.frame_rate:
            audio2 = audio2.set_frame_rate(audio1.frame_rate)
        if audio1.sample_width != audio2.sample_width:
            audio2 = audio2.set_sample_width(audio1.sample_width)
        if audio1.channels != audio2.channels:
            if audio1.channels == 1 and audio2.channels > 1:
                audio2 = audio2.set_channels(1)
            elif audio1.channels > 1 and audio2.channels == 1:
                audio1 = audio1.set_channels(1)

        spliced_audio = audio1 + audio2

        specific_output_dir = os.path.join(output_dir, unique_id)
        os.makedirs(specific_output_dir, exist_ok=True)

        output_filename = f"spliced_{unique_id}.wav"
        output_filepath = os.path.join(specific_output_dir, output_filename)

        spliced_audio.export(output_filepath, format="wav")

        print(f"Successfully spliced {os.path.basename(audio_file_1_path)} and {os.path.basename(audio_file_2_path)}\n              and saved to {output_filepath}")
        return output_filepath
    except Exception as e:
        print(f"Error splicing audio files {audio_file_1_path} and {audio_file_2_path}: {e}")
        return None


Total augmented audio files found for splicing: 6400


In [ ]:
SPLICE_OUTPUT_DIR = "/content/drive/MyDrive/Capstone Project/timit_spliced"
os.makedirs(SPLICE_OUTPUT_DIR, exist_ok=True)

n_spliced_files = 6400 # As per task, same as augmented files count

print(f"Generating {n_spliced_files} spliced audio files...")

# Ensure random selections are distinct if running the whole pipeline from scratch
import random
random.seed(42) # For reproducibility

for i in range(n_spliced_files):
    # Randomly select two distinct audio files
    file1_path, file2_path = random.sample(augmented_audio_files, 2)

    unique_id = f"splice_{i+1}"

    print(f"Splicing operation {i+1}/{n_spliced_files}")
    splice_and_save_audio(file1_path, file2_path, SPLICE_OUTPUT_DIR, unique_id)

print(f"Finished generating {n_spliced_files} spliced audio files.")


Generating 6400 spliced audio files...
Splicing operation 1/6400
Successfully spliced sx377_aug_39.wav and si954_aug_32.wav
              and saved to /content/drive/MyDrive/Capstone Project/timit_spliced/splice_1/spliced_splice_1.wav
Splicing operation 2/6400
Successfully spliced sa1_aug_4.wav and sa2_aug_35.wav
              and saved to /content/drive/MyDrive/Capstone Project/timit_spliced/splice_2/spliced_splice_2.wav
Splicing operation 3/6400
Successfully spliced sx90_aug_15.wav and si1411_aug_8.wav
              and saved to /content/drive/MyDrive/Capstone Project/timit_spliced/splice_3/spliced_splice_3.wav
Splicing operation 4/6400
Successfully spliced sx68_aug_29.wav and sa2_aug_24.wav
              and saved to /content/drive/MyDrive/Capstone Project/timit_spliced/splice_4/spliced_splice_4.wav
Splicing operation 5/6400
Successfully spliced sx296_aug_33.wav and si2214_aug_40.wav
              and saved to /content/drive/MyDrive/Capstone Project/timit_spliced/splice_5/spliced_sp

KeyboardInterrupt: 

### Verify Spliced Files Count

In [ ]:
import os

SPLICE_OUTPUT_DIR = "/content/drive/MyDrive/Capstone Project/timit_spliced"

total_spliced_files = 0
expected_spliced_files = 6400

for root, dirs, files in os.walk(SPLICE_OUTPUT_DIR):
    for file in files:
        if file.endswith(".wav"):
            total_spliced_files += 1

print(f"Total spliced audio files found: {total_spliced_files}")

if total_spliced_files == expected_spliced_files:
    print(f"Verification successful: {total_spliced_files} files match the expected {expected_spliced_files} files.")
else:
    print(f"Verification failed: Found {total_spliced_files} files, but expected {expected_spliced_files} files. (Note: If this count is incorrect, you may need to re-run the splicing process in cell `dbec94ea` if you intend to have 6400 spliced files.)")


Total spliced audio files found: 6400
Verification successful: 6400 files match the expected 6400 files.


# 5. Dataset Creation and Labeling

In [ ]:
import pandas as pd

# 1. Label original audio files as 'real'
real_audio_labels = [(file_path, 'real') for file_path in audio_files]
print(f"Generated {len(real_audio_labels)} labels for original audio files.")

# 2. Re-scan and label augmented audio files as 'fake'
augmented_audio_files_final = []
for root, dirs, files in os.walk(AUGMENTED_VOX_PATH):
    for file in files:
        if file.endswith(".wav"):
            augmented_audio_files_final.append(os.path.join(root, file))
augmented_audio_labels = [(file_path, 'fake') for file_path in augmented_audio_files_final]
print(f"Generated {len(augmented_audio_labels)} labels for augmented audio files.")

# 3. Re-scan and label spliced audio files as 'fake'
spliced_audio_files_final = []
for root, dirs, files in os.walk(SPLICE_OUTPUT_DIR):
    for file in files:
        if file.endswith(".wav"):
            spliced_audio_files_final.append(os.path.join(root, file))
spliced_audio_labels = [(file_path, 'fake') for file_path in spliced_audio_files_final]
print(f"Generated {len(spliced_audio_labels)} labels for spliced audio files.")

# Combine all labels into a single DataFrame
all_audio_data = real_audio_labels + augmented_audio_labels + spliced_audio_labels
df_audio_dataset = pd.DataFrame(all_audio_data, columns=['file_path', 'label'])

# Save the DataFrame to a CSV file
output_csv_path = "audio_dataset.csv"
df_audio_dataset.to_csv(output_csv_path, index=False)

print(f"\nCreated dataset with {len(df_audio_dataset)} entries.")
print(f"DataFrame saved to {output_csv_path}")
print("First 5 rows of the dataset:")
display(df_audio_dataset.head())

# Summary counts
real_count = df_audio_dataset[df_audio_dataset['label'] == 'real'].shape[0]
fake_count = df_audio_dataset[df_audio_dataset['label'] == 'fake'].shape[0]
print(f"\nTotal 'Real' files: {real_count}")
print(f"Total 'Fake' files: {fake_count}")
print(f"Total entries: {real_count + fake_count}")


Generated 160 labels for original audio files.
Generated 6400 labels for augmented audio files.
Generated 6400 labels for spliced audio files.

Created dataset with 12960 entries.
DataFrame saved to audio_dataset.csv
First 5 rows of the dataset:


,file_path,label
0,/content/drive/MyDrive/Capstone Project/timit/...,real
1,/content/drive/MyDrive/Capstone Project/timit/...,real
2,/content/drive/MyDrive/Capstone Project/timit/...,real
3,/content/drive/MyDrive/Capstone Project/timit/...,real
4,/content/drive/MyDrive/Capstone Project/timit/...,real



Total 'Real' files: 160
Total 'Fake' files: 12800
Total entries: 12960


# 6. Dataset Splitting (Train/Validation/Test)

In [ ]:
import pandas as pd
from collections import defaultdict

def get_speaker_id(path):
    """Extracts a speaker-like ID from a given file path, considering original, augmented, and spliced files."""
    # Access global variables defined earlier in the notebook
    global VOX_PATH, AUGMENTED_VOX_PATH, SPLICE_OUTPUT_DIR

    # Handle original TIMIT paths: /content/drive/MyDrive/Capstone Project/timit/timit/drX-XXXXX/...
    if VOX_PATH and VOX_PATH in path:
        # Get the part after the VOX_PATH
        relative_path = path.split(VOX_PATH + '/')[-1]
        path_components = relative_path.split('/')
        if len(path_components) > 0:
            return path_components[0] # Should be the drX-XXXXX part

    # Handle augmented TIMIT paths: /content/drive/MyDrive/Capstone Project/timit_augmented/drX-XXXXX/...
    elif AUGMENTED_VOX_PATH and AUGMENTED_VOX_PATH in path:
        relative_path = path.split(AUGMENTED_VOX_PATH + '/')[-1]
        path_components = relative_path.split('/')
        if len(path_components) > 0:
            return path_components[0] # Should be the drX-XXXXX part

    # Handle spliced paths: /content/drive/MyDrive/Capstone Project/timit_spliced/splice_X/...
    elif SPLICE_OUTPUT_DIR and SPLICE_OUTPUT_DIR in path:
        relative_path = path.split(SPLICE_OUTPUT_DIR + '/')[-1]
        path_components = relative_path.split('/')
        if len(path_components) > 0:
            # For spliced files, the "speaker" is the unique splice_X directory
            # This ensures each splice group is treated distinctly for splitting purposes.
            return path_components[0]

    return "unknown_speaker_id" # Fallback if no known pattern matches

# Reload the full dataset for grouping
df_audio_dataset = pd.read_csv("audio_dataset.csv")

speaker_groups = defaultdict(list)

for index, row in df_audio_dataset.iterrows():
    filepath = row['file_path']
    label = row['label']
    spk = get_speaker_id(filepath)
    speaker_groups[spk].append((filepath, label))

print("Total unique speakers found:", len(speaker_groups))

# Display some speaker groups to verify
print("\nFirst 5 speaker groups (speaker_id: [(filepath, label), ...]):")
for i, (speaker_id, files_and_labels) in enumerate(speaker_groups.items()):
    if i >= 5:
        break
    print(f"{speaker_id}: {files_and_labels[:2]} ... ({len(files_and_labels)} entries)")

Total unique speakers found: 6416

First 5 speaker groups (speaker_id: [(filepath, label), ...]):
dr7-madd0: [('/content/drive/MyDrive/Capstone Project/timit/timit/dr7-madd0/sa2.wav', 'real'), ('/content/drive/MyDrive/Capstone Project/timit/timit/dr7-madd0/si538.wav', 'real')] ... (410 entries)
dr5-mbgt0: [('/content/drive/MyDrive/Capstone Project/timit/timit/dr5-mbgt0/si1341.wav', 'real'), ('/content/drive/MyDrive/Capstone Project/timit/timit/dr5-mbgt0/si1841.wav', 'real')] ... (410 entries)
dr6-mbma1: [('/content/drive/MyDrive/Capstone Project/timit/timit/dr6-mbma1/si2214.wav', 'real'), ('/content/drive/MyDrive/Capstone Project/timit/timit/dr6-mbma1/sx414.wav', 'real')] ... (410 entries)
dr5-ftlg0: [('/content/drive/MyDrive/Capstone Project/timit/timit/dr5-ftlg0/si840.wav', 'real'), ('/content/drive/MyDrive/Capstone Project/timit/timit/dr5-ftlg0/si483.wav', 'real')] ... (410 entries)
dr7-fblv0: [('/content/drive/MyDrive/Capstone Project/timit/timit/dr7-fblv0/sx158.wav', 'real'), ('/c

In [ ]:
from sklearn.model_selection import train_test_split

# Load the combined dataset from the CSV file
df_audio_dataset = pd.read_csv("audio_dataset.csv")

X = df_audio_dataset['file_path']
y = df_audio_dataset['label']

# Split into training (70%) and temp (30% for validation + test)
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, random_state=42, stratify=y)

# Split temp (30%) into validation (15%) and test (15%)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

df_train = pd.DataFrame({'file_path': X_train, 'label': y_train}).reset_index(drop=True)
df_val = pd.DataFrame({'file_path': X_val, 'label': y_val}).reset_index(drop=True)
df_test = pd.DataFrame({'file_path': X_test, 'label': y_test}).reset_index(drop=True)

# Save the split datasets to CSV files
df_train.to_csv('train_dataset.csv', index=False)
df_val.to_csv('val_dataset.csv', index=False)
df_test.to_csv('test_dataset.csv', index=False)

print(f"Training set size: {len(df_train)} samples")
print(f"Validation set size: {len(df_val)} samples")
print(f"Test set size: {len(df_test)} samples")

print("Datasets saved to train_dataset.csv, val_dataset.csv, and test_dataset.csv")


Training set size: 9072 samples
Validation set size: 1944 samples
Test set size: 1944 samples
Datasets saved to train_dataset.csv, val_dataset.csv, and test_dataset.csv


# 7. Feature Extraction and DataLoaders

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
import librosa
import numpy as np
from tqdm.notebook import tqdm # Import tqdm for progress bars

# --- Define extract_features function ---
def extract_features(audio_path, feature_type="lfcc", sr=16000, n_fft=512, hop_length=160, n_mels=60, n_lfcc=60):
    """Extracts specified features from an audio file."""
    max_len = 501 # Fixed length for features

    try:
        # Load audio file
        audio, current_sr = librosa.load(audio_path, sr=sr, mono=True)

        # Handle very short audio files that might not produce any frames
        if len(audio) < hop_length: # Minimum length to produce at least one frame
            return np.zeros((n_lfcc, max_len), dtype=np.float32) # Return a placeholder LFCC array

        if feature_type == "lfcc":
            # Compute Mel Spectrogram
            mel_spectrogram = librosa.feature.melspectrogram(
                y=audio, sr=current_sr, n_fft=n_fft, hop_length=hop_length, n_mels=n_mels
            )

            # Add a small epsilon to avoid log of zero, ensuring numerical stability
            mel_spectrogram = np.maximum(mel_spectrogram, np.finfo(np.float32).eps)

            # Convert to log scale
            log_mel_spectrogram = librosa.power_to_db(mel_spectrogram, ref=1.0)

            # Handle potential -inf values that might still occur (e.g., if eps is too small or other edge cases)
            log_mel_spectrogram[np.isinf(log_mel_spectrogram)] = np.finfo(np.float32).min / 2 # Replace -inf with a large negative number
            log_mel_spectrogram[np.isnan(log_mel_spectrogram)] = 0 # Replace NaN with zero

            # Check if log_mel_spectrogram is empty or has invalid shape
            if log_mel_spectrogram.size == 0 or log_mel_spectrogram.shape[1] == 0:
                return np.zeros((n_lfcc, max_len), dtype=np.float32) # Return a placeholder LFCC array

            # Ensure enough frames for MFCC computation
            min_frames_for_mfcc = n_lfcc # A common practice is to have at least n_mfcc frames
            if log_mel_spectrogram.shape[1] < min_frames_for_mfcc:
                return np.zeros((n_lfcc, max_len), dtype=np.float32)

            # Compute MFCCs (acting as LFCCs)
            lfccs = librosa.feature.mfcc(S=log_mel_spectrogram, sr=current_sr, n_mfcc=n_lfcc, dct_type=2)

            # Pad or truncate to a fixed length
            if lfccs.shape[1] < max_len:
                pad_width = max_len - lfccs.shape[1]
                lfccs = np.pad(lfccs, ((0, 0), (0, pad_width)), mode='constant')
            elif lfccs.shape[1] > max_len:
                lfccs = lfccs[:, :max_len]

            return lfccs.astype(np.float32)
        else:
            raise ValueError(f"Unsupported feature type: {feature_type}")

    except Exception as e:
        print(f"Error processing {audio_path}: {e}. Returning zeros.")
        return np.zeros((n_lfcc, max_len), dtype=np.float32) # (n_lfcc, max_len) as default

# --- Define SpliceDataset class ---
class SpliceDataset(Dataset):
    def __init__(self, dataframe, feature_type="lfcc"):
        self.data = dataframe
        self.feature_type = feature_type
        self.label_map = {'real': 0, 'fake': 1} # Map string labels to numerical values

        self.features = []
        self.labels = []

        print(f"Preprocessing features for {len(dataframe)} audio files...")
        for idx, row in tqdm(dataframe.iterrows(), total=len(dataframe), desc="Extracting features"):
            file_path = row['file_path']
            label_str = row['label']

            feat = extract_features(file_path, self.feature_type)
            self.features.append(torch.tensor(feat, dtype=torch.float32).unsqueeze(0)) # Add channel dim
            self.labels.append(torch.tensor(self.label_map[label_str], dtype=torch.long)) # Convert string label to numerical label
        print("Feature preprocessing complete.")

    def __len__(self):
        return len(self.features)

    def __getitem__(self, idx):
        return self.features[idx], self.labels[idx]

# --- Create DataLoader instances ---

df_train = pd.read_csv('train_dataset.csv')
df_val = pd.read_csv('val_dataset.csv')
df_test = pd.read_csv('test_dataset.csv')

# Temporarily reduce dataset size for faster iteration
sample_size_train = int(len(df_train) * 0.05) # 5% of training data
sample_size_val = int(len(df_val) * 0.1)     # 10% of validation data
sample_size_test = int(len(df_test) * 0.1)    # 10% of test data

df_train = df_train.sample(n=sample_size_train, random_state=42).reset_index(drop=True)
df_val = df_val.sample(n=sample_size_val, random_state=42).reset_index(drop=True)
df_test = df_test.sample(n=sample_size_test, random_state=42).reset_index(drop=True)


train_dataset = SpliceDataset(df_train)
val_dataset   = SpliceDataset(df_val)
test_dataset  = SpliceDataset(df_test)

batch_size = 16

# Use num_workers > 0 for faster data loading in parallel (if system supports it)
# pin_memory=True can speed up data transfer to GPU
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)

print(f"Train Dataset size: {len(train_dataset)}")
print(f"Validation Dataset size: {len(val_dataset)}")
print(f"Test Dataset size: {len(test_dataset)}")

# --- Sanity Check ---
x, y = next(iter(train_loader))
print("\nSanity Check:")
print("Input shape:", x.shape)   # expected [B,1,60,501]
print("Label shape:", y.shape)
print("Sample Input (first batch item):\n", x[0])
print("Sample Label (first batch item):\n", y[0])

Preprocessing features for 453 audio files...


Extracting features:   0%|          | 0/453 [00:00<?, ?it/s]

Feature preprocessing complete.
Preprocessing features for 194 audio files...


Extracting features:   0%|          | 0/194 [00:00<?, ?it/s]

Feature preprocessing complete.
Preprocessing features for 194 audio files...


Extracting features:   0%|          | 0/194 [00:00<?, ?it/s]

Feature preprocessing complete.
Train Dataset size: 453
Validation Dataset size: 194
Test Dataset size: 194

Sanity Check:
Input shape: torch.Size([16, 1, 60, 501])
Label shape: torch.Size([16])
Sample Input (first batch item):
 tensor([[[-3.8588e+01, -3.1811e+01, -9.8082e+01,  ..., -4.2733e+01,
          -2.8483e+01, -1.2946e+01],
         [ 1.2011e+01,  1.0497e+01,  1.6882e+01,  ...,  5.1967e+01,
           3.3415e+01,  1.5007e+01],
         [ 1.9577e+01,  1.8340e+01,  1.5974e+01,  ...,  7.3561e+00,
           2.4836e+01,  1.7009e+01],
         ...,
         [-7.6183e-01, -1.1304e+00, -8.9694e-01,  ...,  6.7833e-01,
          -5.9855e-01,  1.1263e+00],
         [ 9.3647e-01,  5.4367e-01,  7.0615e-01,  ...,  1.0091e+00,
          -9.5944e-01,  3.0173e+00],
         [ 4.5776e-02, -8.5819e-02, -5.0991e-01,  ..., -1.6179e+00,
          -1.6535e+00, -8.3101e-01]]])
Sample Label (first batch item):
 tensor(1)


# 8. Model Definition (`SERes2NetConformer`)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# --- Helper Modules ---

class Res2Block(nn.Module):
    def __init__(self, in_channels, scales=4):
        super(Res2Block, self).__init__()
        self.scales = scales
        width = in_channels // scales

        self.convs = nn.ModuleList()
        self.bns = nn.ModuleList()
        for i in range(scales - 1):
            self.convs.append(nn.Conv2d(width, width, kernel_size=3, padding=1, bias=False))
            self.bns.append(nn.BatchNorm2d(width))
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        spx = torch.split(x, x.size(1) // self.scales, 1)

        sps = []
        for i in range(self.scales):
            if i == 0:
                sps.append(spx[i])
            elif i == 1:
                out_block = self.convs[i-1](spx[i])
                out_block = self.bns[i-1](out_block)
                out_block = self.relu(out_block)
                sps.append(out_block)
            else:
                out_block = spx[i] + sps[-1]
                out_block = self.convs[i-1](out_block)
                out_block = self.bns[i-1](out_block)
                out_block = self.relu(out_block)
                sps.append(out_block)
        return torch.cat(sps, 1)


class ConformerBlock(nn.Module):
    def __init__(self, dim, num_heads, ff_expansion_factor=4, conv_kernel_size=31, dropout=0.1):
        super(ConformerBlock, self).__init__()
        self.dim = dim
        self.num_heads = num_heads

        self.ffn1 = nn.Sequential(
            nn.LayerNorm(dim),
            nn.Linear(dim, dim * ff_expansion_factor),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(dim * ff_expansion_factor, dim),
            nn.Dropout(dropout)
        )

        self.self_attn = nn.Sequential(
            nn.LayerNorm(dim),
            nn.MultiheadAttention(embed_dim=dim, num_heads=num_heads, dropout=dropout, batch_first=True),
            nn.Dropout(dropout)
        )

        self.conv_norm = nn.LayerNorm(dim)

        self.conv_layers = nn.Sequential(
            nn.Conv1d(dim, dim * 2, kernel_size=1),
            nn.GELU(),
            nn.Conv1d(dim * 2, dim * 2, kernel_size=conv_kernel_size, padding=(conv_kernel_size - 1) // 2, groups=dim),
            nn.GELU(),
            nn.Conv1d(dim * 2, dim, kernel_size=1),
            nn.Dropout(dropout)
        )

        self.ffn2 = nn.Sequential(
            nn.LayerNorm(dim),
            nn.Linear(dim, dim * ff_expansion_factor),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(dim * ff_expansion_factor, dim),
            nn.Dropout(dropout)
        )

        self.final_layer_norm = nn.LayerNorm(dim)

    def forward(self, x, mask=None):
        x = x + 0.5 * self.ffn1(x)

        attn_output, _ = self.self_attn[1](x, x, x, attn_mask=mask)
        x = x + attn_output
        x = self.self_attn[2](x)

        residual_conv = x
        x_normalized_for_conv = self.conv_norm(x)
        x_permuted_for_conv = x_normalized_for_conv.permute(0, 2, 1)
        conv_output = self.conv_layers(x_permuted_for_conv)
        x = residual_conv + conv_output.permute(0, 2, 1)

        x = x + 0.5 * self.ffn2(x)

        return self.final_layer_norm(x)


class SERes2NetConformer(nn.Module):
    def __init__(self, conformer_heads=4, conformer_ff_expansion=4, conformer_conv_kernel_size=31, dropout=0.3):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 64, 3, padding=1)
        self.res2 = Res2Block(64)
        self.pool = nn.MaxPool2d(2)
        self.project = nn.Linear(64 * 30, 128)

        self.conformer = ConformerBlock(
            dim=128,
            num_heads=conformer_heads,
            ff_expansion_factor=conformer_ff_expansion,
            conv_kernel_size=conformer_conv_kernel_size,
            dropout=dropout
        )

        self.classifier = nn.Sequential(
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, 2)
        )

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = self.res2(x)
        x = self.pool(x)

        b, c, h, w = x.shape
        x = x.view(b, w, c * h)

        x = self.project(x)

        x = self.conformer(x)

        x = x.mean(dim=1)

        return self.classifier(x)

# Instantiate Model
model = SERes2NetConformer()

print("SERes2NetConformer Model Architecture:")
print(model)


SERes2NetConformer Model Architecture:
SERes2NetConformer(
  (conv1): Conv2d(1, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (res2): Res2Block(
    (convs): ModuleList(
      (0-2): 3 x Conv2d(16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    )
    (bns): ModuleList(
      (0-2): 3 x BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (relu): ReLU(inplace=True)
  )
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (project): Linear(in_features=1920, out_features=128, bias=True)
  (conformer): ConformerBlock(
    (ffn1): Sequential(
      (0): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
      (1): Linear(in_features=128, out_features=512, bias=True)
      (2): GELU(approximate='none')
      (3): Dropout(p=0.3, inplace=False)
      (4): Linear(in_features=512, out_features=128, bias=True)
      (5): Dropout(p=0.3, inplace=False)
    )
    (self_attn): Sequential(
    

# 9. Loss Function and Optimizer Setup

In [ ]:
import torch.nn as nn

# Define Loss Function
criterion = nn.CrossEntropyLoss()

# Define Optimizer
optimizer = torch.optim.Adam(model.parameters(), lr=0.0005) # Lowered learning rate for stability

# Set up Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

print(f"Loss function defined: {criterion}")
print(f"Optimizer defined: {optimizer}")
print(f"Model moved to device: {device}")
if device.type == 'cpu':
    print("Warning: Running on CPU. Training may be very slow. Check 'Runtime' -> 'Change runtime type' to select T4 GPU.")

Loss function defined: CrossEntropyLoss()
Optimizer defined: Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.0005
    maximize: False
    weight_decay: 0
)
Model moved to device: cuda


## 10. Model Training and Evaluation

In [ ]:
from tqdm.notebook import tqdm
import torch.optim.lr_scheduler as lr_scheduler
from torch.cuda.amp import GradScaler
from torch.amp import autocast

def train_model(model, train_loader, val_loader, criterion, optimizer, device, num_epochs=10, scheduler=None, scaler=None, model_save_path="best_model.pth"):
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
    best_val_acc = -1.0

    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        correct_predictions = 0
        total_samples = 0

        train_loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs} Training")
        for inputs, labels in train_loop:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()

            with autocast('cuda', enabled=scaler is not None):
                outputs = model(inputs)
                loss = criterion(outputs, labels)

            if torch.isnan(loss) or torch.isinf(loss):
                print(f"Warning: Loss is NaN/Inf in Epoch {epoch+1}. Skipping batch.")
                continue

            if scaler is not None:
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
                scaler.step(optimizer)
                scaler.update()
            else:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
                optimizer.step()

            running_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs.data, 1)
            total_samples += labels.size(0)
            correct_predictions += (predicted == labels).sum().item()
            train_loop.set_postfix(loss=loss.item(), acc=correct_predictions/total_samples if total_samples > 0 else 0)

        epoch_train_loss = running_loss / total_samples if total_samples > 0 else 0
        epoch_train_acc = correct_predictions / total_samples if total_samples > 0 else 0
        history['train_loss'].append(epoch_train_loss)
        history['train_acc'].append(epoch_train_acc)

        model.eval()
        val_running_loss, val_correct, val_total = 0.0, 0, 0
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                val_running_loss += loss.item() * inputs.size(0)
                _, predicted = torch.max(outputs.data, 1)
                val_total += labels.size(0)
                val_correct += (predicted == labels).sum().item()

        epoch_val_loss = val_running_loss / val_total if val_total > 0 else 0
        epoch_val_acc = val_correct / val_total if val_total > 0 else 0
        history['val_loss'].append(epoch_val_loss)
        history['val_acc'].append(epoch_val_acc)

        if scheduler is not None:
            scheduler.step(epoch_val_loss) if isinstance(scheduler, lr_scheduler.ReduceLROnPlateau) else scheduler.step()

        if epoch_val_acc > best_val_acc:
            best_val_acc = epoch_val_acc
            torch.save(model.state_dict(), model_save_path)

        print(f"Epoch {epoch+1} - Train Loss: {epoch_train_loss:.4f}, Val Acc: {epoch_val_acc:.4f}")

    return model, history

def evaluate_model(model, data_loader, criterion, device, scaler=None):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for inputs, labels in data_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            total_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    return total_loss/total, correct/total

# 10. Model Training

In [59]:
# Re-initializing model to ensure a fresh start after previous errors
model = SERes2NetConformer().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.0005)

# Train the model
num_epochs = 15
model_save_path = "best_seres2netconformer.pth"

# Initialize GradScaler for mixed precision training
scaler = torch.amp.GradScaler('cuda')

# Initialize a learning rate scheduler
scheduler = lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)

print(f"Starting training for {num_epochs} epochs...")
trained_model, training_history = train_model(
    model,
    train_loader,
    val_loader,
    criterion,
    optimizer,
    device,
    num_epochs=num_epochs,
    scheduler=scheduler,
    scaler=scaler,
    model_save_path=model_save_path
)
print("Training finished!")

Starting training for 15 epochs...


Epoch 1/15 Training:   0%|          | 0/29 [00:00<?, ?it/s]

Epoch 1 - Train Loss: 0.1009, Val Acc: 0.9794


Epoch 2/15 Training:   0%|          | 0/29 [00:00<?, ?it/s]

Epoch 2 - Train Loss: 0.0076, Val Acc: 0.9948


...

Epoch 13 - Train Loss: 0.0002, Val Acc: 0.9948


Epoch 14/15 Training:   0%|          | 0/29 [00:00<?, ?it/s]

Epoch 14 - Train Loss: 0.0002, Val Acc: 0.9948


Epoch 15/15 Training:   0%|          | 0/29 [00:00<?, ?it/s]

Epoch 15 - Train Loss: 0.0002, Val Acc: 0.9948
Training finished!


In [60]:
torch.save(
    model.state_dict(),
    "/content/drive/MyDrive/Capstone Project/best_seres2netconformer.pth"
)

In [61]:
import os
print(os.path.exists("/content/drive/MyDrive/Capstone Project/best_seres2netconformer.pth"))

True


In [62]:
torch.save(model.state_dict(), "best_model.pth")

In [63]:
from google.colab import files
files.download("best_model.pth")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [64]:
torch.save(model, "/content/drive/MyDrive/Capstone Project/full_model.pth")

In [65]:
# Evaluate on the test set
print("Evaluating on the test set...")
test_loss, test_accuracy = evaluate_model(trained_model, test_loader, criterion, device, scaler=scaler) # Pass scaler to evaluation

print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")

Evaluating on the test set...
Test Loss: 0.0135
Test Accuracy: 0.9948


In [66]:
from pyngrok import ngrok

ngrok.set_auth_token("3CQiZNxJQjeh3rQZpF65cGEdGPI_5c5WgALHfrBrz2Qf51Az7")

In [68]:
app.run(port=5000)

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit


In [69]:
!ls /content/drive/MyDrive/Capstone\ Project/best_seres2netconformer.pth


'/content/drive/MyDrive/Capstone Project/best_seres2netconformer.pth'


In [70]:
torch.save(
    model.state_dict(),
    "/content/drive/MyDrive/Capstone Project/best_seres2netconformer.pth"
)

In [71]:
import os
print(os.path.exists("/content/drive/MyDrive/Capstone Project/best_seres2netconformer.pth"))

True


In [72]:
model_save_path = "/content/drive/MyDrive/Capstone Project/best_seres2netconformer.pth"

model = SERes2NetConformer().to(device)
model.load_state_dict(torch.load(model_save_path, map_location=device))
model.eval()

SERes2NetConformer(
  (conv1): Conv2d(1, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (res2): Res2Block(
    (convs): ModuleList(
      (0-2): 3 x Conv2d(16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    )
    (bns): ModuleList(
      (0-2): 3 x BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (relu): ReLU(inplace=True)
  )
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (project): Linear(in_features=1920, out_features=128, bias=True)
  (conformer): ConformerBlock(
    (ffn1): Sequential(
      (0): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
      (1): Linear(in_features=128, out_features=512, bias=True)
      (2): GELU(approximate='none')
      (3): Dropout(p=0.3, inplace=False)
      (4): Linear(in_features=512, out_features=128, bias=True)
      (5): Dropout(p=0.3, inplace=False)
    )
    (self_attn): Sequential(
      (0): LayerNorm((128,), eps=1e-05, ele

In [73]:
from google.colab import files
files.download("/content/drive/MyDrive/Capstone Project/best_seres2netconformer.pth")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Loading Model


In [74]:
model = SERes2NetConformer().to(device)

model.load_state_dict(
    torch.load(
        "/content/drive/MyDrive/Capstone Project/best_seres2netconformer.pth",
        map_location=device
    )
)

model.eval()

SERes2NetConformer(
  (conv1): Conv2d(1, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (res2): Res2Block(
    (convs): ModuleList(
      (0-2): 3 x Conv2d(16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    )
    (bns): ModuleList(
      (0-2): 3 x BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (relu): ReLU(inplace=True)
  )
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (project): Linear(in_features=1920, out_features=128, bias=True)
  (conformer): ConformerBlock(
    (ffn1): Sequential(
      (0): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
      (1): Linear(in_features=128, out_features=512, bias=True)
      (2): GELU(approximate='none')
      (3): Dropout(p=0.3, inplace=False)
      (4): Linear(in_features=512, out_features=128, bias=True)
      (5): Dropout(p=0.3, inplace=False)
    )
    (self_attn): Sequential(
      (0): LayerNorm((128,), eps=1e-05, ele

In [75]:
print("Model loaded successfully")

Model loaded successfully


Flask